In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

secrets = UserSecretsClient()

hf_token = secrets.get_secret("HF_TOKEN")
github_token = secrets.get_secret("GITHUB_TOKEN")

login(token=hf_token)

print("Hugging Face authentication configured.")
print("GitHub token retrieved securely.")

In [ ]:
from datasets import load_dataset

english_ds = load_dataset(
    "ai4bharat/Svarah",
    token=hf_token
)

telugu_ds = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    token=hf_token
)

print("ENGLISH")
print(english_ds)

print("\nTELUGU")
print(telugu_ds)

In [ ]:

# PHASE 1: DATASET INSPECTION

!pip install -q datasets huggingface_hub soundfile

import torch
import numpy as np
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

# Authentication
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

# Load datasets
print("Loading English Svarah...")
english_ds = load_dataset(
    "ai4bharat/Svarah",
    token=hf_token
)

print("\nLoading Telugu IndicTTS...")
telugu_ds = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    token=hf_token
)

# Dataset inspection function
def inspect_dataset(name, dataset):
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    total_rows = 0

    for split in dataset:
        data = dataset[split]
        total_rows += len(data)

        print(f"\nSplit: {split}")
        print("Number of samples:", len(data))
        print("Columns:", data.column_names)
        print("Features:", data.features)

        sample = data[0]

        for key, value in sample.items():
            if key.lower() in ["audio", "speech"]:
                if isinstance(value, dict):
                    print(f"{key}:")
                    print("  Sampling rate:", value.get("sampling_rate"))
                    print("  Audio path:", value.get("path"))
                    arr = value.get("array")
                    if arr is not None:
                        print("  Audio shape:", np.asarray(arr).shape)
                else:
                    print(f"{key}: {type(value)}")
            else:
                print(f"{key}: {str(value)[:300]}")

    print("\nTotal samples:", total_rows)


inspect_dataset("ENGLISH - SVARAH", english_ds)
inspect_dataset("TELUGU - INDICTTS", telugu_ds)

# GPU information
print("\n" + "=" * 60)
print("HARDWARE")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")


In [ ]:

import numpy as np
from collections import Counter

# ---------------------------------
# ENGLISH: SVARAH
# ---------------------------------

eng = english_ds["test"]

print("=" * 55)
print("ENGLISH DATA ANALYSIS")
print("=" * 55)

eng_duration = np.array(eng["duration"], dtype=float)

print("Total samples:", len(eng))
print("Total duration:", round(eng_duration.sum() / 3600, 3), "hours")
print("Minimum duration:", round(eng_duration.min(), 2), "seconds")
print("Maximum duration:", round(eng_duration.max(), 2), "seconds")
print("Average duration:", round(eng_duration.mean(), 2), "seconds")

print("\nGender distribution:")
print(Counter(eng["gender"]))

print("\nNative language distribution:")
print(Counter(eng["primary_language"]))

print("\nTranscript examples:")
for i in range(3):
    print(f"{i+1}. {eng[i]['text']}")

# Check available audio decoder
print("\nEnglish audio decoder:")
audio = eng[0]["audio_filepath"]

try:
    samples = audio.get_all_samples()
    print("Sampling rate:", samples.sample_rate)
    print("Audio shape:", samples.data.shape)
except Exception as e:
    print("Audio decoding issue:", repr(e))


# ---------------------------------
# TELUGU: INDICTTS
# ---------------------------------

tel = telugu_ds["train"]

print("\n" + "=" * 55)
print("TELUGU DATA ANALYSIS")
print("=" * 55)

print("Total samples:", len(tel))

print("\nGender distribution:")
print(Counter(tel["gender"]))

print("\nTranscript examples:")
for i in range(3):
    print(f"{i+1}. {tel[i]['text']}")

print("\nTelugu audio inspection:")

for i in range(3):
    audio = tel[i]["audio"]

    try:
        samples = audio.get_all_samples()
        duration = samples.data.shape[-1] / samples.sample_rate

        print(
            f"Sample {i+1}:",
            "Sampling rate:", samples.sample_rate,
            "| Duration:", round(duration, 2), "seconds",
            "| Shape:", tuple(samples.data.shape)
        )
    except Exception as e:
        print(f"Sample {i+1} decoding issue:", repr(e))


# ---------------------------------
# DATA QUALITY
# ---------------------------------

print("\n" + "=" * 55)
print("TEXT QUALITY")
print("=" * 55)

for name, dataset in [
    ("English", eng),
    ("Telugu", tel)
]:
    texts = dataset["text"]

    empty = sum(
        1 for t in texts
        if not isinstance(t, str) or not t.strip()
    )

    print(name)
    print("Empty transcripts:", empty)
    print("Average characters:", round(
        np.mean([len(t) for t in texts if isinstance(t, str)]), 2
    ))


In [ ]:
import io
import numpy as np
import soundfile as sf
from datasets import Audio
from tqdm.auto import tqdm

# Read audio metadata without decoding the complete waveforms
tel_meta = telugu_ds["train"].cast_column(
    "audio", Audio(decode=False)
)

tel_durations = []

for row in tqdm(tel_meta, desc="Checking Telugu durations"):
    audio = row["audio"]

    if audio.get("bytes") is not None:
        info = sf.info(io.BytesIO(audio["bytes"]))
    elif audio.get("path") is not None:
        info = sf.info(audio["path"])
    else:
        continue

    tel_durations.append(info.frames / info.samplerate)

tel_durations = np.array(tel_durations)

print("Telugu total duration:", round(tel_durations.sum() / 3600, 3), "hours")
print("Minimum:", round(tel_durations.min(), 2), "seconds")
print("Maximum:", round(tel_durations.max(), 2), "seconds")
print("Average:", round(tel_durations.mean(), 2), "seconds")

print("\nRecording length counts:")
for threshold in [1, 2, 5, 10, 20, 30]:
    print(f"Under {threshold}s:", np.sum(tel_durations < threshold))

In [ ]:
import numpy as np
import soundfile as sf
import io
from tqdm.auto import tqdm

tel = telugu_ds["train"]
texts = tel["text"]

char_lengths = np.array([len(t) for t in texts])
word_lengths = np.array([len(t.split()) for t in texts])

# Text length relative to recording duration
chars_per_sec = char_lengths / np.maximum(tel_durations, 0.01)

print("TEXT LENGTH")
for q in [25, 50, 75, 90, 95, 99]:
    print(f"{q}th percentile:", round(np.percentile(char_lengths, q), 1))

print("\nWORDS PER RECORDING")
print("Average:", round(word_lengths.mean(), 1))
print("Maximum:", word_lengths.max())

print("\nCHARACTERS PER SECOND")
print("Median:", round(np.median(chars_per_sec), 2))
print("10th percentile:", round(np.percentile(chars_per_sec, 10), 2))
print("90th percentile:", round(np.percentile(chars_per_sec, 90), 2))

# Inspect original audio metadata
tel_meta = tel.cast_column("audio", Audio(decode=False))

sample_rates = []
channels = []

for row in tqdm(tel_meta, desc="Inspecting audio metadata"):
    a = row["audio"]
    source = io.BytesIO(a["bytes"]) if a.get("bytes") else a["path"]
    info = sf.info(source)
    sample_rates.append(info.samplerate)
    channels.append(info.channels)

from collections import Counter

print("\nSAMPLE RATE DISTRIBUTION:", Counter(sample_rates))
print("CHANNEL DISTRIBUTION:", Counter(channels))

# Inspect the 10 longest recordings
print("\n10 LONGEST RECORDINGS")
for i in np.argsort(tel_durations)[-10:][::-1]:
    print(
        f"\nDuration: {tel_durations[i]:.2f}s | "
        f"Characters: {char_lengths[i]} | "
        f"Words: {word_lengths[i]}"
    )
    print(texts[i][:400])

In [ ]:
%pip -q install -U "datasets[audio]" soundfile librosa huggingface_hub tqdm matplotlib

In [ ]:
import datasets
import huggingface_hub
import matplotlib
import torch

print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Matplotlib:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
print("English:", len(english_ds["test"]))
print("Telugu:", len(telugu_ds["train"]))

In [ ]:
import os, io, json, random, csv, re, collections
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
from datasets import load_dataset, Audio
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient

SEED=2026
TARGET_TRAIN=5*3600
TARGET_VAL=1*3600
SR=22050
ROOT=Path('/kaggle/working/fastspeech2_project')
ROOT.mkdir(parents=True,exist_ok=True)
TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
assert TOKEN, 'Set the HF_TOKEN Kaggle secret first'
print('Working directory:',ROOT)


In [ ]:
english=load_dataset('ai4bharat/Svarah', token=TOKEN)['test'].cast_column('audio_filepath', Audio(decode=False))
telugu=load_dataset('SPRINGLab/IndicTTS_Telugu', token=TOKEN)['train'].cast_column('audio', Audio(decode=False))
print('English:',len(english),'Telugu:',len(telugu))
print('English columns:',english.column_names)
print('Telugu columns:',telugu.column_names)

In [ ]:
def audio_info(audio):
    src=io.BytesIO(audio['bytes']) if audio.get('bytes') is not None else audio['path']
    return sf.info(src)

def build_catalog(ds, audio_col, text_col, lang, min_sec, max_sec):
    out=[]; rejected=collections.Counter()
    for i,row in enumerate(tqdm(ds,desc=f'Catalog {lang}')):
        text=str(row[text_col]).strip()
        if not text:
            rejected['empty_text']+=1; continue
        try:
            info=audio_info(row[audio_col]); duration=info.frames/info.samplerate
        except Exception:
            rejected['audio_read_error']+=1; continue
        if not min_sec <= duration <= max_sec:
            rejected['duration_filter']+=1; continue
        if info.channels != 1:
            rejected['not_mono']+=1; continue
        out.append({'index':i,'seconds':duration,'text':text,
                    'gender':str(row.get('gender','unknown')),
                    'source_sr':info.samplerate})
    print(lang,'eligible:',len(out),'eligible hours:',round(sum(x['seconds'] for x in out)/3600,2), 'rejected:',dict(rejected))
    return out

eng_catalog=build_catalog(english,'audio_filepath','text','English',1.0,20.0)
tel_catalog=build_catalog(telugu,'audio','text','Telugu',2.0,20.0)

In [ ]:
def split_by_duration(catalog, seed, train_target=TARGET_TRAIN, val_target=TARGET_VAL):
    rng=random.Random(seed)
    items=catalog.copy(); rng.shuffle(items)
    train=[]; val=[]; train_s=0.; val_s=0.
    # Disjoint record IDs, with a small overshoot permitted at the final recording.
    for x in items:
        if train_s < train_target:
            train.append(x); train_s+=x['seconds']
        elif val_s < val_target:
            val.append(x); val_s+=x['seconds']
        else: break
    if train_s < train_target or val_s < val_target:
        raise RuntimeError(f'Insufficient eligible audio: train={train_s/3600:.2f}h, val={val_s/3600:.2f}h. Increase maximum clip duration cautiously.')
    assert set(x['index'] for x in train).isdisjoint(x['index'] for x in val)
    return train,val

eng_train,eng_val=split_by_duration(eng_catalog,SEED)
tel_train,tel_val=split_by_duration(tel_catalog,SEED+1)
for lang,train,val in [('English',eng_train,eng_val),('Telugu',tel_train,tel_val)]:
    print(lang, 'train:',len(train),'clips',round(sum(x['seconds'] for x in train)/3600,3),'hours',
          '| validation:',len(val),'clips',round(sum(x['seconds'] for x in val)/3600,3),'hours')

In [ ]:
def read_audio(blob):
    src=io.BytesIO(blob['bytes']) if blob.get('bytes') is not None else blob['path']
    audio,sr=sf.read(src,dtype='float32',always_2d=False)
    if audio.ndim==2: audio=audio.mean(axis=1)
    if sr != SR: audio=librosa.resample(audio,orig_sr=sr,target_sr=SR)
    return np.asarray(audio,dtype=np.float32)

def export_language(lang, ds, audio_col, train, val):
    base=ROOT/lang
    stats={}
    for split,records in [('train',train),('val',val)]:
        wav_dir=base/'wavs'/split; wav_dir.mkdir(parents=True,exist_ok=True)
        manifest=base/f'{split}_manifest.csv'
        with manifest.open('w',encoding='utf-8',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=['id','source_index','wav','text','gender','source_seconds','export_seconds','source_sr'])
            writer.writeheader()
            for rec in tqdm(records,desc=f'Export {lang}/{split}'):
                idx=rec['index']; uid=f'{lang.lower()}_{idx:06d}'
                wav_path=wav_dir/f'{uid}.wav'
                if not wav_path.exists():
                    audio=read_audio(ds[idx][audio_col]); sf.write(wav_path,audio,SR,subtype='PCM_16')
                else:
                    audio=None
                exported=sf.info(wav_path).duration
                writer.writerow({'id':uid,'source_index':idx,'wav':str(wav_path),
                    'text':rec['text'],'gender':rec['gender'],
                    'source_seconds':round(rec['seconds'],5),'export_seconds':round(exported,5),
                    'source_sr':rec['source_sr']})
        stats[split]={'count':len(records),'hours':sum(x['seconds'] for x in records)/3600,
                      'manifest':str(manifest)}
    return stats

stats={}
stats['English']=export_language('English',english,'audio_filepath',eng_train,eng_val)
stats['Telugu']=export_language('Telugu',telugu,'audio',tel_train,tel_val)
with (ROOT/'data_audit.json').open('w',encoding='utf-8') as f:
    json.dump({'seed':SEED,'target_train_hours':5,'target_val_hours':1,
               'sample_rate':SR,'note':'English uses internal split of source test split; speaker disjointness not guaranteed; Telugu alignment not yet performed',
               'stats':stats},f,indent=2,ensure_ascii=False)
print(json.dumps(stats,indent=2))

In [ ]:
# Save a compact metadata-only archive (WAVs remain in /kaggle/working).
import zipfile
archive=ROOT.parent/'fastspeech2_manifests.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if p.is_file() and p.suffix in ('.csv','.json'):
            z.write(p,p.relative_to(ROOT.parent))
print('Metadata archive:',archive)
print('Audio directory:',ROOT)
print('NEXT: obtain trustworthy phoneme/grapheme duration alignments before running FastSpeech 2 training.')

In [ ]:
from pathlib import Path
import json
from collections import Counter

root = Path("/kaggle/working/fastspeech2_project")

for lang in ["English", "Telugu"]:
    folder = root / lang
    wavs = list(folder.rglob("*.wav"))
    print(f"{lang}: {len(wavs)} WAV files")

print("\nProject structure:")
for p in sorted(root.iterdir()):
    print(p.name)

print("\nManifest files:")
for p in root.rglob("*"):
    if p.is_file() and p.suffix in [".json", ".jsonl", ".csv", ".txt"]:
        print(p.relative_to(root))

In [ ]:
import json
from pathlib import Path

p = Path("/kaggle/working/fastspeech2_project/data_audit.json")

with open(p, encoding="utf-8") as f:
    audit = json.load(f)

print(json.dumps(audit, indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/working/fastspeech2_project")

for lang in ["English", "Telugu"]:
    for split in ["train", "val"]:
        path = root / lang / f"{split}_manifest.csv"
        df = pd.read_csv(path)

        print(f"\n{lang} / {split}")
        print("Columns:", df.columns.tolist())
        print("First 2 records:")
        print(df.head(2).to_string(index=False, max_colwidth=65))

In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import json

ROOT = Path("/kaggle/working/fastspeech2_project")
ALIGN = ROOT / "alignment"

summary = {}

for lang in ["English", "Telugu"]:
    summary[lang] = {}

    for split in ["train", "val"]:
        df = pd.read_csv(ROOT / lang / f"{split}_manifest.csv")

        # Separate corpus for each language and split
        corpus = ALIGN / lang / split
        corpus.mkdir(parents=True, exist_ok=True)

        completed = 0

        for row in df.itertuples(index=False):
            source = Path(row.wav)

            if not source.exists():
                raise FileNotFoundError(source)

            # Link rather than duplicate large audio files
            destination = corpus / f"{row.id}.wav"

            if not destination.exists():
                destination.symlink_to(source)

            # MFA reads transcript from matching .lab file
            transcript = corpus / f"{row.id}.lab"
            transcript.write_text(
                str(row.text).strip(),
                encoding="utf-8"
            )

            completed += 1

        summary[lang][split] = completed
        print(f"{lang} {split}: {completed} alignment pairs")

(ALIGN / "corpus_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\nAlignment corpus prepared:", ALIGN)

In [ ]:
%pip install -q ctc-forced-aligner

In [ ]:
import torchaudio

print("AVAILABLE ASR BUNDLES:\n")

for name in dir(torchaudio.pipelines):
    if any(x in name for x in ["WAV2VEC2", "MMS", "XLSR"]):
        print(name)

In [ ]:
import torch
import torchaudio

device = "cuda" if torch.cuda.is_available() else "cpu"

bundle = torchaudio.pipelines.MMS_FA

print("Model sample rate:", bundle.sample_rate)
print("Model labels:", bundle.get_labels())

model = bundle.get_model().to(device)
model.eval()

print("MMS forced-alignment model loaded on:", device)

In [ ]:
labels = bundle.get_labels()

print("Vocabulary size:", len(labels))
print("First 50 labels:", labels[:50])

print("\nTelugu characters in model vocabulary:")
telugu_chars = [
    "అ", "ఆ", "ఇ", "క", "గ", "త", "న", "మ", "ర", "ల", "వ"
]

for ch in telugu_chars:
    print(ch, ":", ch in labels)

In [ ]:
%pip install -q uroman

In [ ]:
from uroman import Uroman

romanizer = Uroman()
labels = set(bundle.get_labels())

examples = [
    "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
    "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
    "పులి అడవిలో నిశ్శబ్దంగా నడిచింది."
]

for sentence in examples:
    converted = romanizer.romanize_string(sentence, lcode="tel")
    print("\nOriginal:", sentence)
    print("Romanized:", converted)
    print("Unsupported characters:",
          sorted(set(converted.lower()) - labels - {" "}))

In [ ]:
import pandas as pd
import torch
import torchaudio
import re
import unicodedata
from uroman import Uroman

root = "/kaggle/working/fastspeech2_project"
df = pd.read_csv(f"{root}/Telugu/train_manifest.csv")

# Use a short recording for the first test
row = df[df["export_seconds"].between(2, 6)].iloc[0]

wav, sr = torchaudio.load(row["wav"])
wav = wav.mean(dim=0, keepdim=True)

if sr != bundle.sample_rate:
    wav = torchaudio.functional.resample(
        wav, sr, bundle.sample_rate
    )

romanizer = Uroman()

romanized = romanizer.romanize_string(
    str(row["text"]), lcode="tel"
).lower()

# Retain only characters supported by MMS_FA
labels = bundle.get_labels()
allowed = set(labels) - {"-", "*"}
normalized = "".join(
    c if c in allowed or c.isspace() else " "
    for c in romanized
)
normalized = " ".join(normalized.split())

# MMS_FA uses * as the word separator
normalized = normalized.replace(" ", "*")

label_to_id = {c: i for i, c in enumerate(labels)}
tokens = [label_to_id[c] for c in normalized]

print("Recording:", row["id"])
print("Duration:", round(wav.shape[-1] / bundle.sample_rate, 2), "seconds")
print("Original:", row["text"])
print("Normalized:", normalized)
print("Number of tokens:", len(tokens))

with torch.inference_mode():
    emissions, _ = model(wav.to(device))
    emissions = torch.log_softmax(emissions, dim=-1)

targets = torch.tensor([tokens], dtype=torch.int32, device=device)
lengths = torch.tensor(
    [emissions.shape[1]],
    dtype=torch.int64,
    device=device
)

alignments, scores = torchaudio.functional.forced_align(
    emissions,
    targets,
    input_lengths=lengths,
    blank=0
)

spans = torchaudio.functional.merge_tokens(
    alignments[0].cpu(),
    scores[0].cpu(),
    blank=0
)

seconds_per_frame = (
    wav.shape[-1] / bundle.sample_rate / emissions.shape[1]
)

print("\nFIRST 20 ALIGNED TOKENS")
for token, span in zip(normalized[:20], spans[:20]):
    print(
        token,
        f"{span.start * seconds_per_frame:.3f}s",
        "→",
        f"{span.end * seconds_per_frame:.3f}s",
        "score:",
        round(span.score, 3)
    )

print("\nAlignment spans:", len(spans))
print("Expected tokens:", len(tokens))

In [ ]:
import torch
import numpy as np

with torch.inference_mode():
    probs = emissions.exp()[0]
    confidence, predicted_ids = probs.max(dim=-1)

print("Mean maximum probability:", round(confidence.mean().item(), 4))
print("Median maximum probability:", round(confidence.median().item(), 4))

print("\nMost frequently predicted tokens:")
counts = torch.bincount(
    predicted_ids.cpu(),
    minlength=len(labels)
)

for idx in counts.argsort(descending=True)[:10]:
    print(
        repr(labels[idx]),
        int(counts[idx]),
        "frames"
    )

print("\nEmission shape:", tuple(emissions.shape))
print("Audio duration:", round(wav.shape[-1] / bundle.sample_rate, 2), "seconds")

In [ ]:
# Use the model's original emissions directly
emissions = raw_emissions

with torch.inference_mode():
    alignments, scores = torchaudio.functional.forced_align(
        emissions,
        targets,
        input_lengths=lengths,
        blank=0
    )

spans = torchaudio.functional.merge_tokens(
    alignments[0].cpu(),
    scores[0].cpu(),
    blank=0
)

seconds_per_frame = (
    wav.shape[-1] / bundle.sample_rate / emissions.shape[1]
)

print("Expected tokens:", len(tokens))
print("Aligned spans:", len(spans))

print("\nFIRST 20 ALIGNMENTS")
for token, span in zip(normalized[:20], spans[:20]):
    print(
        token,
        f"{span.start * seconds_per_frame:.3f}s",
        "→",
        f"{span.end * seconds_per_frame:.3f}s",
        "score:",
        round(span.score, 3)
    )

# Inspect model predictions without changing emissions
predicted_ids = emissions[0].argmax(dim=-1)
counts = torch.bincount(
    predicted_ids.cpu(),
    minlength=len(labels)
)

print("\nTOP PREDICTED TOKENS")
for idx in counts.argsort(descending=True)[:10]:
    print(repr(labels[idx]), int(counts[idx]), "frames")

In [ ]:
import torch

labels = bundle.get_labels()

print("Token IDs:")
for i, label in enumerate(labels):
    print(i, repr(label))

print("\nModel output:")
print("Shape:", tuple(raw_emissions.shape))
print("NaN values:", torch.isnan(raw_emissions).any().item())
print("Infinite values:", torch.isinf(raw_emissions).any().item())

print("\nTop tokens at selected frames:")
for frame in [0, 20, 50, 85, 120, 160]:
    values, indices = raw_emissions[0, frame].topk(5)
    print(
        f"Frame {frame}:",
        [(labels[i], round(v, 3))
         for i, v in zip(indices.tolist(), values.tolist())]
    )

print("\nAlignment path token counts:")
path = alignments[0].cpu()
for i in torch.unique(path):
    print(repr(labels[i.item()]), int((path == i).sum()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

audio = wav.squeeze().cpu().numpy()
time = np.arange(len(audio)) / bundle.sample_rate

fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(time, audio, linewidth=0.4, alpha=0.7)

# Show aligned words as colored time regions
word_start = None
word_chars = []

for char, span in zip(normalized, spans):
    if char == "*":
        if word_chars and word_start is not None:
            word = "".join(word_chars)
            word_end = span.start * seconds_per_frame

            ax.axvspan(
                word_start,
                word_end,
                alpha=0.12
            )

            ax.text(
                (word_start + word_end) / 2,
                0.75,
                word,
                rotation=45,
                ha="center",
                transform=ax.get_xaxis_transform(),
                fontsize=8
            )

        word_start = None
        word_chars = []

    else:
        if word_start is None:
            word_start = span.start * seconds_per_frame

        word_chars.append(char)

# Final word
if word_chars and word_start is not None:
    ax.axvspan(
        word_start,
        spans[-1].end * seconds_per_frame,
        alpha=0.12
    )

ax.set_title("Telugu MMS Forced Alignment — First Recording")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Amplitude")
ax.set_xlim(0, len(audio) / bundle.sample_rate)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import torch
import torchaudio
import re

df = pd.read_csv(
    "/kaggle/working/fastspeech2_project/Telugu/train_manifest.csv"
)

samples = df[df["export_seconds"].between(2, 10)].head(10)

results = []

for row in samples.itertuples(index=False):
    try:
        audio, sr = torchaudio.load(row.wav)
        audio = audio.mean(dim=0, keepdim=True)

        if sr != bundle.sample_rate:
            audio = torchaudio.functional.resample(
                audio, sr, bundle.sample_rate
            )

        roman = romanizer.romanize_string(
            str(row.text), lcode="tel"
        ).lower()

        clean = "".join(
            c if c in allowed or c.isspace() else " "
            for c in roman
        )
        clean = "*".join(clean.split())

        ids = [label_to_id[c] for c in clean]

        with torch.inference_mode():
            em, _ = model(audio.to(device))

            target = torch.tensor(
                [ids], dtype=torch.int32, device=device
            )

            lengths = torch.tensor(
                [em.shape[1]], dtype=torch.int64, device=device
            )

            alignment, scores = torchaudio.functional.forced_align(
                em,
                target,
                input_lengths=lengths,
                blank=0
            )

            spans = torchaudio.functional.merge_tokens(
                alignment[0].cpu(),
                scores[0].cpu(),
                blank=0
            )

        results.append({
            "id": row.id,
            "duration": round(audio.shape[-1] / bundle.sample_rate, 2),
            "tokens": len(ids),
            "spans": len(spans),
            "status": "OK" if len(spans) == len(ids) else "MISMATCH"
        })

    except Exception as e:
        results.append({
            "id": row.id,
            "status": f"ERROR: {str(e)[:100]}"
        })

print(pd.DataFrame(results).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import torchaudio
import json
import re
import gc
from tqdm.auto import tqdm

ROOT = Path("/kaggle/working/fastspeech2_project")
OUT = ROOT / "alignments"
OUT.mkdir(exist_ok=True)

labels = bundle.get_labels()
label_to_id = {c: i for i, c in enumerate(labels)}
allowed = set(labels) - {"-", "*"}

def normalize_text(text, lang):
    text = str(text)

    if lang == "Telugu":
        text = romanizer.romanize_string(text, lcode="tel")

    text = text.lower()
    text = "".join(
        c if c in allowed or c.isspace() else " "
        for c in text
    )

    return "*".join(text.split())

def align_record(row, lang):
    audio, sr = torchaudio.load(row.wav)
    audio = audio.mean(dim=0, keepdim=True)

    if sr != bundle.sample_rate:
        audio = torchaudio.functional.resample(
            audio, sr, bundle.sample_rate
        )

    normalized = normalize_text(row.text, lang)

    if not normalized:
        raise ValueError("Empty normalized transcript")

    token_ids = [label_to_id[c] for c in normalized]

    with torch.inference_mode():
        emissions, _ = model(audio.to(device))

        targets = torch.tensor(
            [token_ids],
            dtype=torch.int32,
            device=device
        )

        input_lengths = torch.tensor(
            [emissions.shape[1]],
            dtype=torch.int64,
            device=device
        )

        alignment, scores = torchaudio.functional.forced_align(
            emissions,
            targets,
            input_lengths=input_lengths,
            blank=0
        )

        spans = torchaudio.functional.merge_tokens(
            alignment[0].cpu(),
            scores[0].cpu(),
            blank=0
        )

    if len(spans) != len(token_ids):
        raise ValueError(
            f"Token/span mismatch: {len(token_ids)} vs {len(spans)}"
        )

    seconds_per_frame = (
        audio.shape[-1] / bundle.sample_rate / emissions.shape[1]
    )

    return {
        "id": row.id,
        "language": lang,
        "wav": row.wav,
        "original_text": str(row.text),
        "normalized_text": normalized,
        "audio_seconds": audio.shape[-1] / bundle.sample_rate,
        "tokens": [
            {
                "char": char,
                "start": round(span.start * seconds_per_frame, 5),
                "end": round(span.end * seconds_per_frame, 5),
                "score": float(span.score)
            }
            for char, span in zip(normalized, spans)
        ]
    }

model.eval()

for lang in ["English", "Telugu"]:
    for split in ["train", "val"]:
        manifest = pd.read_csv(
            ROOT / lang / f"{split}_manifest.csv"
        )

        output = OUT / f"{lang.lower()}_{split}.jsonl"
        errors = OUT / f"{lang.lower()}_{split}_errors.jsonl"

        completed = set()

        if output.exists():
            with open(output, encoding="utf-8") as f:
                for line in f:
                    try:
                        completed.add(json.loads(line)["id"])
                    except Exception:
                        pass

        print(f"\n{lang} / {split}: {len(manifest)} recordings")
        print(f"Already completed: {len(completed)}")

        with open(output, "a", encoding="utf-8") as success_file, \
             open(errors, "a", encoding="utf-8") as error_file:

            for row in tqdm(
                manifest.itertuples(index=False),
                total=len(manifest),
                desc=f"{lang} {split}"
            ):
                if row.id in completed:
                    continue

                try:
                    result = align_record(row, lang)

                    success_file.write(
                        json.dumps(result, ensure_ascii=False) + "\n"
                    )
                    success_file.flush()

                except Exception as e:
                    error_file.write(
                        json.dumps({
                            "id": row.id,
                            "error": str(e)
                        }, ensure_ascii=False) + "\n"
                    )
                    error_file.flush()

        print("Saved:", output)

print("\nBATCH ALIGNMENT FINISHED")

In [ ]:
from pathlib import Path
import json

folder = Path("/kaggle/working/fastspeech2_project/alignments")

for lang, train_total, val_total in [
    ("english", 3097, 613),
    ("telugu", 1895, 358),
]:
    for split, expected in [
        ("train", train_total),
        ("val", val_total),
    ]:
        success = folder / f"{lang}_{split}.jsonl"
        errors = folder / f"{lang}_{split}_errors.jsonl"

        def count_records(path):
            if not path.exists():
                return 0
            with open(path, encoding="utf-8") as f:
                return sum(1 for line in f if line.strip())

        good = count_records(success)
        bad = count_records(errors)

        print(
            f"{lang}/{split}: "
            f"{good} successful, "
            f"{bad} errors, "
            f"{expected} expected"
        )

        if bad:
            with open(errors, encoding="utf-8") as f:
                for line in list(f)[:2]:
                    print("  Error:", json.loads(line))

In [ ]:
from pathlib import Path
import json
import pandas as pd

root = Path("/kaggle/working/fastspeech2_project")

for split in ["train", "val"]:
    manifest = pd.read_csv(root / "English" / f"{split}_manifest.csv")

    errors = root / "alignments" / f"english_{split}_errors.jsonl"

    failed = []
    with open(errors, encoding="utf-8") as f:
        for line in f:
            failed.append(json.loads(line)["id"])

    rows = manifest[manifest["id"].isin(failed)]

    print(f"\nENGLISH {split.upper()}")
    print("Failed recordings:", len(rows))
    print("Duration lost:", round(rows["export_seconds"].sum() / 60, 2), "minutes")

    for _, row in rows.head(10).iterrows():
        print(repr(row["text"]))

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

root = Path("/kaggle/working/fastspeech2_project")
folder = root / "alignments"

for lang in ["english", "telugu"]:
    for split in ["train", "val"]:
        path = folder / f"{lang}_{split}.jsonl"
        records = []

        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

        problems = []
        coverage = []

        for record in records:
            tokens = record["tokens"]
            duration = record["audio_seconds"]

            if not tokens:
                problems.append((record["id"], "No tokens"))
                continue

            starts = np.array([t["start"] for t in tokens])
            ends = np.array([t["end"] for t in tokens])

            if np.any(ends <= starts):
                problems.append((record["id"], "Zero/negative token duration"))

            if np.any(starts < 0) or np.any(ends > duration + 0.05):
                problems.append((record["id"], "Timestamp outside audio"))

            if np.any(starts[1:] < starts[:-1]):
                problems.append((record["id"], "Non-monotonic alignment"))

            coverage.append(
                (ends[-1] - starts[0]) / max(duration, 0.01)
            )

        print(f"\n{lang.upper()} / {split}")
        print("Successful records:", len(records))
        print("Records with issues:", len(set(p[0] for p in problems)))
        print("Median alignment coverage:",
              round(float(np.median(coverage)) * 100, 1), "%")
        print("First issues:", problems[:5])

**Stage 02 — FastSpeech 2 aligned acoustic features (English + Telugu)**

In [ ]:
import numpy as np, pandas as pd, librosa, soundfile as sf, json, traceback
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
ROOT=Path('/kaggle/working/fastspeech2_project')
FEATURES=ROOT/'features'
SR=22050; HOP=256; N_FFT=1024; WIN=1024; N_MELS=80; FMIN=0; FMAX=8000
for lang in ('English','Telugu'):
    for split in ('train','val'):
        (FEATURES/lang/split).mkdir(parents=True,exist_ok=True)
print('Feature destination:',FEATURES)

In [ ]:
def load_records(lang, split):
    p=ROOT/'alignments'/f'{lang.lower()}_{split}.jsonl'
    with p.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def extract(rec):
    y,sr=sf.read(rec['wav'],dtype='float32',always_2d=False)
    if y.ndim>1:y=y.mean(axis=1)
    if sr!=SR:y=librosa.resample(y,orig_sr=sr,target_sr=SR)
    if len(y)<N_FFT:raise ValueError('Audio shorter than FFT window')
    mel=librosa.feature.melspectrogram(y=y,sr=SR,n_fft=N_FFT,hop_length=HOP,win_length=WIN,
                                      n_mels=N_MELS,fmin=FMIN,fmax=FMAX,power=1.0,center=True)
    mel=np.log(np.maximum(mel,1e-5)).T.astype('float32')
    T=len(mel)
    stft=np.abs(librosa.stft(y,n_fft=N_FFT,hop_length=HOP,win_length=WIN,center=True))
    energy=np.linalg.norm(stft,axis=0).astype('float32')[:T]
    # Align per-token durations to mel-frame centers; allocate every frame to nearest token center.
    ts=rec['tokens']; chars=''.join(t['char'] for t in ts)
    centers=np.array([(t['start']+t['end'])/2 for t in ts],dtype='float64')
    if len(centers)==0 or np.any(np.diff(centers)<0):raise ValueError('Bad token centers')
    bounds=(centers[:-1]+centers[1:])/2
    frame_times=np.arange(T,dtype='float64')*HOP/SR
    owners=np.searchsorted(bounds,frame_times,side='right')
    duration=np.bincount(owners,minlength=len(ts)).astype('int32')
    # YIN produces pitch even on silence; mask low-energy frames as unvoiced.
    f0=librosa.yin(y,sr=SR,fmin=65,fmax=700,frame_length=2048,hop_length=HOP,center=True)
    if len(f0)<T:f0=np.pad(f0,(0,T-len(f0)))
    f0=f0[:T].astype('float32')
    rms=librosa.feature.rms(y=y,frame_length=2048,hop_length=HOP,center=True)[0][:T]
    threshold=max(float(np.percentile(rms,20))*1.5,0.005)
    f0[rms<threshold]=0
    # Token-level averages; non-voiced pitch=0.
    pitch=np.zeros(len(ts),dtype='float32'); tok_energy=np.zeros(len(ts),dtype='float32')
    for i in range(len(ts)):
        idx=(owners==i)
        if idx.any():
            voiced=f0[idx]; voiced=voiced[voiced>0]
            pitch[i]=float(np.mean(voiced)) if len(voiced) else 0
            tok_energy[i]=float(np.mean(energy[idx]))
    assert int(duration.sum())==T
    assert len(chars)==len(duration)==len(pitch)==len(tok_energy)
    assert np.isfinite(mel).all() and np.isfinite(pitch).all() and np.isfinite(tok_energy).all()
    return dict(mel=mel,duration=duration,pitch=pitch,energy=tok_energy,chars=chars)

r=load_records('Telugu','train')[0]
a=extract(r)
print('ID:',r['id'],'Tokens:',len(a['chars']),'Mel frames:',len(a['mel']),
      'Duration sum:',a['duration'].sum(),'Pitch voiced tokens:',int((a['pitch']>0).sum()))
print('Mel shape:',a['mel'].shape)

In [ ]:
for lang in ('English','Telugu'):
    for split in ('train','val'):
        records=load_records(lang,split)
        output_dir=FEATURES/lang/split
        errors=[]; index=[]
        for rec in tqdm(records,desc=f'{lang} {split}'):
            out=output_dir/(rec['id']+'.npz')
            try:
                if not out.exists():
                    d=extract(rec)
                    np.savez_compressed(out,mel=d['mel'],duration=d['duration'],
                                        pitch=d['pitch'],energy=d['energy'])
                with np.load(out) as z:
                    n_frames=len(z['mel']); n_tokens=len(z['duration'])
                    if int(z['duration'].sum())!=n_frames:raise ValueError('Frame mismatch')
                index.append(dict(id=rec['id'],feature_path=str(out),wav=rec['wav'],
                                  original_text=rec['original_text'],text=rec['normalized_text'],
                                  mel_frames=n_frames,tokens=n_tokens))
            except Exception as e:
                errors.append(dict(id=rec['id'],error=str(e)))
        (FEATURES/lang/f'{split}_index.jsonl').write_text(
            ''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in index),encoding='utf-8')
        (FEATURES/lang/f'{split}_errors.json').write_text(json.dumps(errors,ensure_ascii=False,indent=2),encoding='utf-8')
        print(lang,split,'features:',len(index),'errors:',len(errors))
        if errors:print('First errors:',errors[:3])
print('All feature batches attempted')

In [ ]:
for lang in ('English','Telugu'):
    all_rows=[]
    for split in ('train','val'):
        p=FEATURES/lang/f'{split}_index.jsonl'
        with p.open(encoding='utf-8') as f:all_rows.extend(json.loads(x) for x in f if x.strip())
    chars=sorted(set(''.join(r['text'] for r in all_rows)))
    vocab={ch:i+1 for i,ch in enumerate(chars)}
    (FEATURES/lang/'vocab.json').write_text(json.dumps(vocab,ensure_ascii=False,indent=2),encoding='utf-8')
    settings=dict(sample_rate=SR,hop_length=HOP,n_fft=N_FFT,win_length=WIN,n_mels=N_MELS,
                  fmin=FMIN,fmax=FMAX,pitch_method='yin_with_rms_mask',
                  duration_method='nearest_token_center_partition',
                  text_representation='romanized_character',vocab_size=len(vocab)+1)
    (FEATURES/lang/'feature_config.json').write_text(json.dumps(settings,indent=2),encoding='utf-8')
    print(lang,'records:',len(all_rows),'vocab size including PAD:',len(vocab)+1,
          'characters:',repr(''.join(chars)))
print('NEXT: FastSpeech 2 model + separate training loops; do not confuse these features with a trained model.')

**Stage 03 — Train independent FastSpeech 2-style models**> 

In [ ]:
import os, json, math, random, time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT = Path('/kaggle/working/fastspeech2_project')
FEATURES = ROOT / 'features'
CKPT = ROOT / 'checkpoints'
CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Device:', DEVICE)
assert (FEATURES/'English'/'train_index.jsonl').exists()
assert (FEATURES/'Telugu'/'train_index.jsonl').exists()

class AcousticDataset(Dataset):
    def __init__(self, lang, split):
        self.rows = [json.loads(x) for x in (FEATURES/lang/f'{split}_index.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
        self.vocab = json.loads((FEATURES/lang/'vocab.json').read_text(encoding='utf-8'))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        with np.load(r['feature_path']) as z:
            mel = torch.from_numpy(z['mel'].astype('float32'))
            dur = torch.from_numpy(z['duration'].astype('int64'))
            pitch = torch.from_numpy(z['pitch'].astype('float32'))
            energy = torch.from_numpy(z['energy'].astype('float32'))
        ids = torch.tensor([self.vocab[c] for c in r['text']], dtype=torch.long)
        if len(ids) != len(dur) or int(dur.sum()) != len(mel):
            raise ValueError(f'Bad feature dimensions: {r["id"]}')
        return ids, dur, pitch, energy, mel

def collate(batch):
    B = len(batch)
    nt = max(len(x[0]) for x in batch); nm = max(len(x[4]) for x in batch)
    tok = torch.zeros(B,nt,dtype=torch.long); dur = torch.zeros(B,nt,dtype=torch.long)
    pitch = torch.zeros(B,nt); energy = torch.zeros(B,nt)
    mel = torch.zeros(B,nm,80); tl = torch.zeros(B,dtype=torch.long); ml = torch.zeros(B,dtype=torch.long)
    for i,(a,d,p,e,m) in enumerate(batch):
        t=len(a); f=len(m); tok[i,:t]=a; dur[i,:t]=d; pitch[i,:t]=p; energy[i,:t]=e; mel[i,:f]=m; tl[i]=t; ml[i]=f
    return tok,dur,pitch,energy,mel,tl,ml

for lang in ('English','Telugu'):
    d=AcousticDataset(lang,'train')
    a=d[0]
    print(lang,'samples:',len(d),'tokens:',len(a[0]),'mel frames:',len(a[4]))

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, hidden, max_len=20000):
        super().__init__()
        pos=torch.arange(max_len).float().unsqueeze(1)
        div=torch.exp(torch.arange(0,hidden,2).float()*(-math.log(10000.0)/hidden))
        pe=torch.zeros(max_len,hidden); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0),persistent=False)
    def forward(self,x): return x+self.pe[:,:x.shape[1]].to(x.dtype)

class Predictor(nn.Module):
    def __init__(self,h):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(h,h,3,padding=1),nn.ReLU(),nn.LayerNorm([h,1]) if False else nn.Identity(),nn.Dropout(.1),
                               nn.Conv1d(h,h,3,padding=1),nn.ReLU(),nn.Dropout(.1),nn.Conv1d(h,1,1))
    def forward(self,x):return self.net(x.transpose(1,2)).squeeze(1)

class FastSpeech2Compact(nn.Module):
    def __init__(self,vocab_size,h=192,layers=3,heads=4):
        super().__init__()
        self.h=h; self.embedding=nn.Embedding(vocab_size,h,padding_idx=0); self.pos=PositionalEncoding(h)
        self.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(h,heads,dim_feedforward=h*4,dropout=.1,batch_first=True,norm_first=True),layers,enable_nested_tensor=False)
        self.duration_predictor=Predictor(h)
        self.pitch_predictor=Predictor(h)
        self.energy_predictor=Predictor(h)
        self.pitch_embedding=nn.Linear(1,h); self.energy_embedding=nn.Linear(1,h)
        self.decoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(h,heads,dim_feedforward=h*4,dropout=.1,batch_first=True,norm_first=True),layers,enable_nested_tensor=False)
        self.mel_linear=nn.Linear(h,80)
        self.postnet=nn.Sequential(nn.Conv1d(80,256,5,padding=2),nn.Tanh(),nn.Conv1d(256,80,5,padding=2))
    def forward(self,tokens,durations=None,pitch=None,energy=None,max_frames=3000):
        mask=tokens.eq(0)
        x=self.encoder(self.pos(self.embedding(tokens)),src_key_padding_mask=mask)
        logdur=self.duration_predictor(x).masked_fill(mask,0)
        predpitch=self.pitch_predictor(x).masked_fill(mask,0)
        predenergy=self.energy_predictor(x).masked_fill(mask,0)
        if durations is None:
            durations=(torch.expm1(logdur).clamp(min=0,max=50).round().long()).masked_fill(mask,0)
            durations=torch.where((durations==0)&(~mask),torch.ones_like(durations),durations)
        if pitch is None:pitch=predpitch
        if energy is None:energy=predenergy
        x=x+self.pitch_embedding((pitch/300).unsqueeze(-1))+self.energy_embedding((energy/100).unsqueeze(-1))
        # Variable-length regulator. Ground-truth durations used in training; predicted durations at inference.
        seq=[torch.repeat_interleave(x[i,:int((~mask[i]).sum())],durations[i,:int((~mask[i]).sum())].clamp(min=0),dim=0)[:max_frames] for i in range(len(x))]
        lens=torch.tensor([len(s) for s in seq],device=x.device)
        if int(lens.min())==0:raise ValueError('Zero mel length from duration regulator')
        z=nn.utils.rnn.pad_sequence(seq,batch_first=True)
        mm=torch.arange(z.shape[1],device=z.device)[None,:]>=lens[:,None]
        z=self.decoder(self.pos(z),src_key_padding_mask=mm)
        mel=self.mel_linear(z); post=mel+self.postnet(mel.transpose(1,2)).transpose(1,2)
        return {'mel':mel,'post_mel':post,'logdur':logdur,'pitch':predpitch,'energy':predenergy,'mel_lengths':lens}

def loss_fn(out,batch):
    tokens,dur,pitch,energy,mel,tl,ml=batch
    tmask=(torch.arange(tokens.shape[1],device=tokens.device)[None,:]<tl[:,None]).float()
    n=min(mel.shape[1],out['mel'].shape[1]); mmask=(torch.arange(n,device=tokens.device)[None,:]<ml[:,None]).unsqueeze(-1).float()
    mel_loss=((out['mel'][:,:n]-mel[:,:n]).abs()*mmask).sum()/(mmask.sum()*80).clamp_min(1)
    post_loss=((out['post_mel'][:,:n]-mel[:,:n]).abs()*mmask).sum()/(mmask.sum()*80).clamp_min(1)
    dl=((out['logdur']-torch.log1p(dur.float()))**2*tmask).sum()/tmask.sum().clamp_min(1)
    pl=(((out['pitch']-pitch)/300)**2*tmask).sum()/tmask.sum().clamp_min(1)
    el=(((out['energy']-energy)/100)**2*tmask).sum()/tmask.sum().clamp_min(1)
    return mel_loss+post_loss+.1*dl+.1*pl+.1*el, (mel_loss.item(),post_loss.item(),dl.item(),pl.item(),el.item())
print('Model defined')

In [ ]:
def train_language(lang, epochs=30, batch_size=8, lr=2e-4, resume=True, max_steps_per_epoch=None):
    train=AcousticDataset(lang,'train'); val=AcousticDataset(lang,'val')
    dl=DataLoader(train,batch_size=batch_size,shuffle=True,collate_fn=collate,num_workers=2,pin_memory=True)
    vl=DataLoader(val,batch_size=batch_size,shuffle=False,collate_fn=collate,num_workers=2,pin_memory=True)
    vocab=json.loads((FEATURES/lang/'vocab.json').read_text(encoding='utf-8'))
    net=FastSpeech2Compact(len(vocab)+1).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=1e-6)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
    outdir=CKPT/lang; outdir.mkdir(parents=True,exist_ok=True)
    last=outdir/'last.pt'; best=outdir/'best.pt'; hist_path=outdir/'history.json'
    history=[]; start=0; bestval=float('inf')
    if resume and last.exists():
        state=torch.load(last,map_location=DEVICE,weights_only=False)
        net.load_state_dict(state['model']); opt.load_state_dict(state['optimizer'])
        start=state['epoch']+1; bestval=state.get('best_val',float('inf'))
        history=json.loads(hist_path.read_text()) if hist_path.exists() else []
        print(lang,'resuming from epoch',start+1)
    for epoch in range(start,epochs):
        net.train(); losses=[]; t0=time.time()
        for step,batch in enumerate(dl):
            if max_steps_per_epoch and step>=max_steps_per_epoch:break
            batch=tuple(x.to(DEVICE,non_blocking=True) for x in batch)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                pred=net(batch[0],batch[1],batch[2],batch[3]); loss,_=loss_fn(pred,batch)
            if not torch.isfinite(loss):raise ValueError('Non-finite training loss')
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(opt); scaler.update(); losses.append(loss.item())
        net.eval(); vloss=[]
        with torch.inference_mode():
            for batch in vl:
                batch=tuple(x.to(DEVICE,non_blocking=True) for x in batch)
                with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                    pred=net(batch[0],batch[1],batch[2],batch[3]); loss,_=loss_fn(pred,batch)
                vloss.append(loss.item())
        tr=float(np.mean(losses)); va=float(np.mean(vloss))
        history.append({'epoch':epoch+1,'train_loss':tr,'val_loss':va})
        state={'epoch':epoch,'model':net.state_dict(),'optimizer':opt.state_dict(),'best_val':min(bestval,va),'vocab':vocab,'config':{'hidden':192,'layers':3,'heads':4,'n_mels':80,'sample_rate':22050,'hop_length':256}}
        torch.save(state,last)
        if va<bestval:bestval=va;torch.save(state,best)
        hist_path.write_text(json.dumps(history,indent=2))
        print(f'{lang} epoch {epoch+1}/{epochs}: train={tr:.4f}, val={va:.4f}, time={time.time()-t0:.0f}s')
        fig,ax=plt.subplots(figsize=(7,4));ax.plot([r['epoch'] for r in history],[r['train_loss'] for r in history],label='Train');ax.plot([r['epoch'] for r in history],[r['val_loss'] for r in history],label='Validation');ax.set(xlabel='Epoch',ylabel='Loss',title=f'{lang} loss');ax.legend();fig.tight_layout();fig.savefig(outdir/'loss.png',dpi=140);plt.close(fig)
    return net,history
print('Training function defined. Checkpoints saved after every epoch.')

In [ ]:
# Test one forward/backward batch WITHOUT saving a checkpoint.
for lang in ('English','Telugu'):
    ds=AcousticDataset(lang,'train')
    batch=collate([ds[0],ds[1]])
    batch=tuple(x.to(DEVICE) for x in batch)
    net=FastSpeech2Compact(len(ds.vocab)+1).to(DEVICE)
    pred=net(batch[0],batch[1],batch[2],batch[3])
    loss,parts=loss_fn(pred,batch)
    loss.backward()
    print(lang,'forward/backward OK','loss',round(loss.item(),4),'mel shape',tuple(pred['mel'].shape))
    del net,pred,loss
if DEVICE.type=='cuda':torch.cuda.empty_cache()

**Full training**
Run English first, then Telugu. Change epochs to 50–100 if runtime allows; monitor validation and use best.pt. This is a compact model and audio quality is not guaranteed by a small dataset or a few epochs. For a Kaggle time limit, run in sessions and resume with epochs set to a larger total epoch count. Do not run both training cells concurrently on one GPU.

In [ ]:
english_model, english_history = train_language('English',epochs=30,batch_size=8,lr=2e-4,resume=True)

In [ ]:
del english_model
if DEVICE.type=='cuda': torch.cuda.empty_cache()
telugu_model, telugu_history = train_language('Telugu',epochs=30,batch_size=8,lr=2e-4,resume=True)

In [ ]:
for lang in ('English','Telugu'):
    p=CKPT/lang
    print(lang,'best checkpoint:',(p/'best.pt').exists(),'last:',(p/'last.pt').exists(),'history:',(p/'history.json').exists())
    if (p/'history.json').exists():
        h=json.loads((p/'history.json').read_text());print('epochs:',len(h),'last:',h[-1])
print('NEXT: Stage 04 inference, waveform vocoder, exact test sentences, variance plots, report.')

**Replacement Stage 03 — FastSpeech2-style v2 (English and Telugu)*# ***

In [ ]:
import os, json, math, time, random, warnings
from pathlib import Path
import numpy as np, torch, librosa, soundfile as sf
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import Audio, display
ROOT=Path('/kaggle/working/fastspeech2_project')
FEATURES=ROOT/'features'
CKPT_V2=ROOT/'checkpoints_v2'
CKPT_V2.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED=2026
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
assert (FEATURES/'English'/'train_index.jsonl').exists()
assert (FEATURES/'Telugu'/'train_index.jsonl').exists()
print('Device:',DEVICE,'| V2 output:',CKPT_V2)


In [ ]:
def load_index(lang,split):
    return [json.loads(x) for x in (FEATURES/lang/f'{split}_index.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]

def get_stats(lang):
    p=CKPT_V2/lang/'stats.json'
    if p.exists():return json.loads(p.read_text())
    p.parent.mkdir(parents=True,exist_ok=True)
    count=0; sm=np.zeros(80,dtype=np.float64); sq=np.zeros(80,dtype=np.float64)
    pv=[]; ev=[]
    rows=load_index(lang,'train')
    for j,r in enumerate(rows):
        with np.load(r['feature_path']) as z:
            mel=z['mel'].astype(np.float64)
            pitch=z['pitch'].astype(np.float64)
            energy=z['energy'].astype(np.float64)
        if mel.shape[1]!=80 or not np.isfinite(mel).all():raise ValueError('Bad mel: '+r['id'])
        sm+=mel.sum(0);sq+=(mel**2).sum(0);count+=len(mel)
        pv.extend(np.log1p(np.clip(pitch[pitch>1],0,None)).tolist())
        ev.extend(np.log1p(np.clip(energy,0,None)).tolist())
    mean=sm/count;std=np.sqrt(np.maximum(sq/count-mean**2,1e-5))
    pmean=float(np.mean(pv));pstd=max(float(np.std(pv)),.15)
    emean=float(np.mean(ev));estd=max(float(np.std(ev)),.15)
    stats={'mel_mean':mean.tolist(),'mel_std':std.tolist(),
           'pitch_mean':pmean,'pitch_std':pstd,
           'energy_mean':emean,'energy_std':estd,
           'num_frames':count,'num_clips':len(rows)}
    p.write_text(json.dumps(stats,indent=2))
    return stats

STATS={lang:get_stats(lang) for lang in ('English','Telugu')}
for lang,s in STATS.items():
    print(lang,'training frames',s['num_frames'],'mel mean range',
          (round(min(s['mel_mean']),2),round(max(s['mel_mean']),2)),
          'pitch log stats',round(s['pitch_mean'],2),round(s['pitch_std'],2))


In [ ]:
class AcousticDatasetV2(Dataset):
    def __init__(self,lang,split):
        self.rows=load_index(lang,split)
        self.vocab=json.loads((FEATURES/lang/'vocab.json').read_text())
        self.s=STATS[lang]
        self.mm=np.array(self.s['mel_mean'],dtype=np.float32)
        self.ms=np.array(self.s['mel_std'],dtype=np.float32)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        with np.load(r['feature_path']) as z:
            mel=z['mel'].astype('float32')
            dur=z['duration'].astype('int64')
            pitch=z['pitch'].astype('float32')
            energy=z['energy'].astype('float32')
        tok=np.array([self.vocab[x] for x in r['text']],dtype=np.int64)
        if len(tok)!=len(dur) or int(dur.sum())!=len(mel):raise ValueError('Bad duration alignment: '+r['id'])
        mel=np.clip((mel-self.mm)/self.ms,-8,8)
        voiced=(pitch>1).astype('float32')
        p=np.where(voiced>0,(np.log1p(np.maximum(pitch,0))-self.s['pitch_mean'])/self.s['pitch_std'],0).astype('float32')
        e=((np.log1p(np.maximum(energy,0))-self.s['energy_mean'])/self.s['energy_std']).astype('float32')
        return tuple(torch.from_numpy(a) for a in (tok,dur,p,e,mel,voiced))

def collate_v2(items):
    B=len(items); nt=max(len(x[0]) for x in items);nf=max(len(x[4]) for x in items)
    tok=torch.zeros(B,nt,dtype=torch.long);dur=torch.zeros(B,nt,dtype=torch.long)
    pit=torch.zeros(B,nt);eng=torch.zeros(B,nt);voi=torch.zeros(B,nt)
    mel=torch.zeros(B,nf,80);tl=torch.zeros(B,dtype=torch.long);ml=torch.zeros(B,dtype=torch.long)
    for i,(t,d,p,e,m,v) in enumerate(items):
        a=len(t);b=len(m)
        tok[i,:a]=t;dur[i,:a]=d;pit[i,:a]=p;eng[i,:a]=e;voi[i,:a]=v
        mel[i,:b]=m;tl[i]=a;ml[i]=b
    return tok,dur,pit,eng,mel,voi,tl,ml

for lang in ('English','Telugu'):
    ds=AcousticDatasetV2(lang,'train');x=ds[0]
    print(lang,'train clips',len(ds),'first mel',tuple(x[4].shape),'dur sum',int(x[1].sum()))


In [ ]:
class AcousticDatasetV2(Dataset):
    def __init__(self,lang,split):
        self.rows=load_index(lang,split)
        self.vocab=json.loads((FEATURES/lang/'vocab.json').read_text())
        self.s=STATS[lang]
        self.mm=np.array(self.s['mel_mean'],dtype=np.float32)
        self.ms=np.array(self.s['mel_std'],dtype=np.float32)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        with np.load(r['feature_path']) as z:
            mel=z['mel'].astype('float32')
            dur=z['duration'].astype('int64')
            pitch=z['pitch'].astype('float32')
            energy=z['energy'].astype('float32')
        tok=np.array([self.vocab[x] for x in r['text']],dtype=np.int64)
        if len(tok)!=len(dur) or int(dur.sum())!=len(mel):raise ValueError('Bad duration alignment: '+r['id'])
        mel=np.clip((mel-self.mm)/self.ms,-8,8)
        voiced=(pitch>1).astype('float32')
        p=np.where(voiced>0,(np.log1p(np.maximum(pitch,0))-self.s['pitch_mean'])/self.s['pitch_std'],0).astype('float32')
        e=((np.log1p(np.maximum(energy,0))-self.s['energy_mean'])/self.s['energy_std']).astype('float32')
        return tuple(torch.from_numpy(a) for a in (tok,dur,p,e,mel,voiced))

def collate_v2(items):
    B=len(items); nt=max(len(x[0]) for x in items);nf=max(len(x[4]) for x in items)
    tok=torch.zeros(B,nt,dtype=torch.long);dur=torch.zeros(B,nt,dtype=torch.long)
    pit=torch.zeros(B,nt);eng=torch.zeros(B,nt);voi=torch.zeros(B,nt)
    mel=torch.zeros(B,nf,80);tl=torch.zeros(B,dtype=torch.long);ml=torch.zeros(B,dtype=torch.long)
    for i,(t,d,p,e,m,v) in enumerate(items):
        a=len(t);b=len(m)
        tok[i,:a]=t;dur[i,:a]=d;pit[i,:a]=p;eng[i,:a]=e;voi[i,:a]=v
        mel[i,:b]=m;tl[i]=a;ml[i]=b
    return tok,dur,pit,eng,mel,voi,tl,ml

for lang in ('English','Telugu'):
    ds=AcousticDatasetV2(lang,'train');x=ds[0]
    print(lang,'train clips',len(ds),'first mel',tuple(x[4].shape),'dur sum',int(x[1].sum()))


In [ ]:
class SinusoidalPosV2(nn.Module):
    def __init__(self,h,max_len=12000):
        super().__init__()
        pos=torch.arange(max_len).float()[:,None]
        div=torch.exp(torch.arange(0,h,2).float()*(-math.log(10000)/h))
        pe=torch.zeros(max_len,h);pe[:,0::2]=torch.sin(pos*div);pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe[None],persistent=False)
    def forward(self,x):return x+self.pe[:,:x.shape[1]].to(x.dtype)

class ConvPredictorV2(nn.Module):
    def __init__(self,h):
        super().__init__()
        self.c1=nn.Conv1d(h,h,3,padding=1)
        self.c2=nn.Conv1d(h,h,3,padding=1)
        self.norm1=nn.LayerNorm(h);self.norm2=nn.LayerNorm(h)
        self.out=nn.Linear(h,1);self.drop=nn.Dropout(.1)
    def forward(self,x):
        x=self.drop(self.norm1(F.relu(self.c1(x.transpose(1,2)).transpose(1,2))))
        x=self.drop(self.norm2(F.relu(self.c2(x.transpose(1,2)).transpose(1,2))))
        return self.out(x).squeeze(-1)

class FastSpeech2V2(nn.Module):
    def __init__(self,vocab_size,h=192,layers=3,heads=4):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,h,padding_idx=0)
        self.pos=SinusoidalPosV2(h)
        def transformer():
            return nn.TransformerEncoder(nn.TransformerEncoderLayer(
                h,heads,dim_feedforward=h*4,dropout=.1,batch_first=True,norm_first=True),
                layers,enable_nested_tensor=False)
        self.encoder=transformer();self.decoder=transformer()
        self.duration_predictor=ConvPredictorV2(h)
        self.pitch_predictor=ConvPredictorV2(h)
        self.energy_predictor=ConvPredictorV2(h)
        self.voicing_predictor=ConvPredictorV2(h)
        self.pitch_embedding=nn.Linear(1,h)
        self.energy_embedding=nn.Linear(1,h)
        self.mel_linear=nn.Linear(h,80)
        self.postnet=nn.Sequential(nn.Conv1d(80,256,5,padding=2),nn.Tanh(),
                                   nn.Conv1d(256,256,5,padding=2),nn.Tanh(),
                                   nn.Conv1d(256,80,5,padding=2))
    def forward(self,tokens,durations=None,pitch=None,energy=None,voiced=None,max_frames=3000):
        mask=tokens.eq(0)
        x=self.encoder(self.pos(self.embedding(tokens)),src_key_padding_mask=mask)
        logdur=self.duration_predictor(x).masked_fill(mask,0)
        pp=self.pitch_predictor(x).masked_fill(mask,0)
        ee=self.energy_predictor(x).masked_fill(mask,0)
        vv=self.voicing_predictor(x).masked_fill(mask,0)
        if durations is None:
            durations=torch.expm1(logdur.float().clamp(0,4.0)).round().long().clamp(1,45).masked_fill(mask,0)
        if pitch is None:pitch=pp*torch.sigmoid(vv).gt(.5).float()
        if energy is None:energy=ee
        x=x+self.pitch_embedding(pitch.unsqueeze(-1))+self.energy_embedding(energy.unsqueeze(-1))
        seq=[]
        for i in range(len(x)):
            k=int((~mask[i]).sum().item())
            s=torch.repeat_interleave(x[i,:k],durations[i,:k].clamp(min=0),dim=0)[:max_frames]
            if len(s)==0:raise ValueError('Zero output length')
            seq.append(s)
        lens=torch.tensor([len(s) for s in seq],device=x.device)
        z=nn.utils.rnn.pad_sequence(seq,batch_first=True)
        m=torch.arange(z.shape[1],device=z.device)[None,:]>=lens[:,None]
        z=self.decoder(self.pos(z),src_key_padding_mask=m)
        mel=self.mel_linear(z)
        post=mel+self.postnet(mel.transpose(1,2)).transpose(1,2)
        return {'mel':mel,'post_mel':post,'logdur':logdur,
                'pitch':pp,'energy':ee,'voicing':vv,'mel_lengths':lens}

def loss_v2(out,b):
    tok,dur,pit,eng,mel,voi,tl,ml=b
    tm=(torch.arange(tok.shape[1],device=tok.device)[None,:]<tl[:,None]).float()
    fm=(torch.arange(mel.shape[1],device=tok.device)[None,:]<ml[:,None]).unsqueeze(-1).float()
    assert out['mel'].shape[1]==mel.shape[1]
    a=((out['mel'].float()-mel).abs()*fm).sum()/(fm.sum()*80).clamp_min(1)
    z=((out['post_mel'].float()-mel).abs()*fm).sum()/(fm.sum()*80).clamp_min(1)
    d=(((out['logdur'].float()-torch.log1p(dur.float()))**2)*tm).sum()/tm.sum().clamp_min(1)
    p=(((out['pitch'].float()-pit)**2)*tm*voi).sum()/(tm*voi).sum().clamp_min(1)
    e=(((out['energy'].float()-eng)**2)*tm).sum()/tm.sum().clamp_min(1)
    v=(F.binary_cross_entropy_with_logits(out['voicing'].float(),voi,reduction='none')*tm).sum()/tm.sum().clamp_min(1)
    total=a+z+.15*d+.05*p+.05*e+.05*v
    return total,{'mel':float(a.detach()),'post':float(z.detach()),'duration':float(d.detach()),
                  'pitch':float(p.detach()),'energy':float(e.detach()),'voicing':float(v.detach())}
print('V2 model and loss defined')


In [ ]:
def v2_mel_to_wav(normalized_mel,stats,n_iter=48):
    a=np.asarray(normalized_mel,dtype='float32')
    raw=a*np.array(stats['mel_std'],dtype='float32')+np.array(stats['mel_mean'],dtype='float32')
    magnitude=np.exp(np.clip(raw.T,-15,12))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        wav=librosa.feature.inverse.mel_to_audio(
            magnitude,sr=22050,n_fft=1024,hop_length=256,win_length=1024,
            fmin=0,fmax=8000,power=1.0,n_iter=n_iter)
    wav=np.nan_to_num(wav).astype('float32')
    peak=np.max(np.abs(wav)) if len(wav) else 0
    if peak>1e-7:wav=wav/peak*.9
    return wav

def preview_v2(lang,net,epoch):
    ds=AcousticDatasetV2(lang,'val')
    idx=next((i for i in range(len(ds)) if 170<=len(ds[i][4])<=340),0)
    tok,dur,pit,eng,mel,voi=ds[idx]
    net.eval()
    with torch.inference_mode():
        t=tok[None].to(DEVICE)
        forced=net(t,dur[None].to(DEVICE),pit[None].to(DEVICE),
                   eng[None].to(DEVICE),voi[None].to(DEVICE))
        free=net(t)
    directory=CKPT_V2/lang/'previews';directory.mkdir(parents=True,exist_ok=True)
    clips={'target':mel.numpy(),
           'teacher':forced['post_mel'][0].float().cpu().numpy(),
           'free':free['post_mel'][0].float().cpu().numpy()}
    for tag,a in clips.items():
        wav=v2_mel_to_wav(a,STATS[lang],n_iter=40)
        sf.write(directory/f'epoch_{epoch:03d}_{tag}.wav',wav,22050)
    print(lang,'epoch',epoch,'preview frames: actual',int(dur.sum()),
          'free',int(free['mel_lengths'][0]))
    return directory


In [ ]:
for lang in ('English','Telugu'):
    ds=AcousticDatasetV2(lang,'train')
    batch=collate_v2([ds[0],ds[1]])
    batch=tuple(x.to(DEVICE) for x in batch)
    net=FastSpeech2V2(len(ds.vocab)+1).to(DEVICE)
    out=net(batch[0],batch[1],batch[2],batch[3],batch[5])
    loss,parts=loss_v2(out,batch)
    loss.backward()
    print(lang,'smoke test OK, loss',round(loss.item(),3),'mel shape',tuple(out['mel'].shape),parts)
    del net,out,loss
    if DEVICE.type=='cuda':torch.cuda.empty_cache()


In [ ]:
def train_language_v2(lang,epochs=40,batch_size=8,lr=2e-4,resume=True,preview_every=5):
    train=AcousticDatasetV2(lang,'train');val=AcousticDatasetV2(lang,'val')
    dl=DataLoader(train,batch_size=batch_size,shuffle=True,collate_fn=collate_v2,num_workers=2,pin_memory=True)
    vl=DataLoader(val,batch_size=batch_size,shuffle=False,collate_fn=collate_v2,num_workers=2,pin_memory=True)
    net=FastSpeech2V2(len(train.vocab)+1).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs,eta_min=lr*.1)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
    folder=CKPT_V2/lang;folder.mkdir(parents=True,exist_ok=True)
    last=folder/'last.pt';best=folder/'best.pt';history_path=folder/'history.json'
    history=[];start=0;bestval=float('inf')
    if resume and last.exists():
        state=torch.load(last,map_location=DEVICE,weights_only=False)
        net.load_state_dict(state['model']);opt.load_state_dict(state['optimizer'])
        if 'scheduler' in state:sched.load_state_dict(state['scheduler'])
        start=state['epoch']+1;bestval=state['best_val']
        if history_path.exists():history=json.loads(history_path.read_text())
        print(lang,'resume at epoch',start+1)
    for ep in range(start,epochs):
        t0=time.time();net.train();tr=[]
        for batch in dl:
            b=tuple(x.to(DEVICE,non_blocking=True) for x in batch)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                out=net(b[0],b[1],b[2],b[3],b[5]);loss,_=loss_v2(out,b)
            if not torch.isfinite(loss):raise ValueError('Nonfinite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.)
            scaler.step(opt);scaler.update();tr.append(float(loss.detach()))
        net.eval();va=[]
        with torch.inference_mode():
            for batch in vl:
                b=tuple(x.to(DEVICE,non_blocking=True) for x in batch)
                with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                    out=net(b[0],b[1],b[2],b[3],b[5]);loss,_=loss_v2(out,b)
                va.append(float(loss.detach()))
        sched.step();t=float(np.mean(tr));v=float(np.mean(va))
        history.append({'epoch':ep+1,'train_loss':t,'val_loss':v,'lr':opt.param_groups[0]['lr']})
        improved=v<bestval
        if improved:bestval=v
        state={'epoch':ep,'model':net.state_dict(),'optimizer':opt.state_dict(),
               'scheduler':sched.state_dict(),'best_val':bestval,'vocab':train.vocab,
               'stats':STATS[lang], 'architecture':'FastSpeech2V2',
               'config':{'hidden':192,'layers':3,'heads':4,'sample_rate':22050,
                         'hop_length':256,'n_mels':80,'mel_target':'normalized_log_magnitude'}}
        torch.save(state,last)
        if improved:torch.save(state,best)
        history_path.write_text(json.dumps(history,indent=2))
        print(f'{lang} V2 epoch {ep+1}/{epochs}: train={t:.4f}, val={v:.4f}, best={bestval:.4f}, {time.time()-t0:.0f}s')
        if (ep+1)%preview_every==0 or ep==0:
            try:preview_v2(lang,net,ep+1)
            except Exception as exc:print('Preview failed (training checkpoint safe):',repr(exc))
        fig,ax=plt.subplots(figsize=(7,4))
        ax.plot([r['epoch'] for r in history],[r['train_loss'] for r in history],label='train')
        ax.plot([r['epoch'] for r in history],[r['val_loss'] for r in history],label='val')
        ax.set(title=f'{lang} v2 loss',xlabel='Epoch',ylabel='Combined loss');ax.legend()
        fig.tight_layout();fig.savefig(folder/'loss.png',dpi=140);plt.close(fig)
    return net,history
print('V2 trainer ready. Old checkpoints are untouched.')


In [ ]:
english_v2, english_v2_history=train_language_v2('English',epochs=40,batch_size=8,lr=2e-4,resume=True)

In [ ]:
del english_v2
if DEVICE.type=='cuda':torch.cuda.empty_cache()
telugu_v2,telugu_v2_history=train_language_v2('Telugu',epochs=40,batch_size=8,lr=2e-4,resume=True)

In [ ]:
from pathlib import Path
from IPython.display import display, Image

OUT = Path("/kaggle/working/fastspeech2_project/training_visualizations")

png = OUT / "fastspeech2_v2_training_dashboard.png"

print("PNG exists:", png.exists())

if png.exists():
    print("File size:", round(png.stat().st_size / 1024 / 1024, 2), "MB")
    display(Image(filename=str(png)))

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/fastspeech2_project")

for lang in ["English", "Telugu"]:
    print(f"\n{lang}")
    print("Train features:",
          len(list((root / "features" / lang / "train").glob("*.npz"))))
    print("Validation features:",
          len(list((root / "features" / lang / "val").glob("*.npz"))))
    print("Best V2 checkpoint:",
          (root / "checkpoints_v2" / lang / "best.pt").exists())

In [ ]:
import inspect
import json
from pathlib import Path

root = Path("/kaggle/working/fastspeech2_project")

print("=== V2 NORMALIZATION FILES ===")
for lang in ["English", "Telugu"]:
    print(f"\n{lang}:")
    for p in (root / "checkpoints_v2" / lang).glob("*"):
        print(" ", p.name)

print("\n=== AUDIO RECONSTRUCTION FUNCTIONS ===")
for name, obj in list(globals().items()):
    if (
        inspect.isfunction(obj)
        and any(word in name.lower() for word in [
            "audio", "vocoder", "inverse", "denorm", "preview"
        ])
    ):
        print("\nFUNCTION:", name)
        try:
            print(inspect.getsource(obj)[:3500])
        except (OSError, TypeError):
            print("Source unavailable")

print("\n=== NORMALIZATION VARIABLES ===")
for name, obj in list(globals().items()):
    if any(word in name.lower() for word in [
        "mel_mean", "mel_std", "norm_stats", "feature_stats"
    ]):
        print(name, type(obj).__name__)

In [ ]:
import inspect
import json
from pathlib import Path

root = Path("/kaggle/working/fastspeech2_project")

print("=== V2 MEL TO WAV FUNCTION ===")
print(inspect.getsource(v2_mel_to_wav))

print("\n=== NORMALIZATION STATISTICS ===")
for lang in ["English", "Telugu"]:
    path = root / "checkpoints_v2" / lang / "stats.json"

    with open(path, encoding="utf-8") as f:
        stats = json.load(f)

    print(f"\n{lang}:")
    for key, value in stats.items():
        if isinstance(value, list):
            print(key, "length:", len(value),
                  "first 5:", value[:5])
        else:
            print(key, value)

print("\n=== V2 DATASET NORMALIZATION ===")
print(inspect.getsource(AcousticDatasetV2.__getitem__))

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/kaggle/working/fastspeech2_project")
OUT = ROOT / "training_visualizations"
OUT.mkdir(parents=True, exist_ok=True)

def compare_mels_v2(lang, net, split="val"):
    ds = AcousticDatasetV2(lang, split)

    idx = next(
        (i for i in range(len(ds)) if 170 <= len(ds[i][4]) <= 340),
        0
    )

    tok, dur, pit, eng, mel, voi = ds[idx]

    net.eval()
    device = next(net.parameters()).device

    with torch.inference_mode():
        tokens = tok[None].to(device)

        forced = net(
            tokens,
            dur[None].to(device),
            pit[None].to(device),
            eng[None].to(device),
            voi[None].to(device)
        )

        free = net(tokens)

    images = [
        ("Ground-truth mel-spectrogram", mel.numpy()),
        ("Teacher-forced prediction", forced["post_mel"][0].cpu().numpy()),
        ("Fully predicted mel-spectrogram", free["post_mel"][0].cpu().numpy())
    ]

    fig, axes = plt.subplots(
        3, 1,
        figsize=(15, 13),
        gridspec_kw={"hspace": 0.55}
    )

    for ax, (heading, data) in zip(axes, images):
        im = ax.imshow(
            data.T,
            aspect="auto",
            origin="lower",
            cmap="magma",
            vmin=-3,
            vmax=3
        )

        ax.set_title(heading, fontsize=13, fontweight="bold", pad=15)
        ax.set_ylabel("Mel frequency bin", labelpad=12)
        ax.set_xlabel("Time frame", labelpad=8)

    # Leave space for the main title and colorbar
    fig.subplots_adjust(
        left=0.09,
        right=0.88,
        top=0.91,
        bottom=0.07
    )

    cbar = fig.colorbar(
        im,
        ax=axes,
        location="right",
        fraction=0.025,
        pad=0.025
    )
    cbar.set_label("Normalized log-mel")

    fig.suptitle(
        f"{lang} — FastSpeech 2 V2 Mel-Spectrogram Comparison",
        fontsize=17,
        fontweight="bold",
        y=0.975
    )

    save_path = OUT / f"{lang.lower()}_v2_mel_comparison.png"

    fig.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {save_path}")
    print(f"Target frames: {len(images[0][1])}")
    print(f"Teacher-forced frames: {len(images[1][1])}")
    print(f"Fully predicted frames: {len(images[2][1])}")

compare_mels_v2("English", english_v2, split="train")
compare_mels_v2("Telugu", telugu_v2, split="train")

save_path = OUT / f"{lang.lower()}_v2_{split}_mel_comparison.png"

**Stage 04 — Final Speech Synthesis and Variance Adaptor Visualizations.**

In [ ]:
import torch

for lang in ["English", "Telugu"]:
    path = ROOT / "checkpoints_v2" / lang / "best.pt"
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    vocab = ckpt["vocab"]

    print(f"\n=== {lang} ===")
    print("Vocabulary type:", type(vocab).__name__)
    print("Vocabulary size:", len(vocab))
    print("First 15 entries:", list(vocab.items())[:15] if isinstance(vocab, dict) else vocab[:15])
    print("Checkpoint epoch:", ckpt["epoch"])
    print("Architecture:", ckpt["architecture"])

In [ ]:
import inspect

print("=== TEXT PROCESSING FUNCTIONS ===")

for name, obj in list(globals().items()):
    if inspect.isfunction(obj) and any(
        x in name.lower()
        for x in ["roman", "uroman", "normalize", "clean", "token", "translit"]
    ):
        print(name, inspect.signature(obj))

print("\n=== VOCABULARY COVERAGE ===")

for lang in ["English", "Telugu"]:
    ckpt = torch.load(
        ROOT / "checkpoints_v2" / lang / "best.pt",
        map_location="cpu",
        weights_only=False
    )

    vocab = ckpt["vocab"]
    print(lang, "characters:", "".join(sorted(vocab.keys())))

In [ ]:
import torch
from pathlib import Path

ROOT = Path("/kaggle/working/fastspeech2_project")

TEST_SENTENCES = {
    "English": {
        "E1": "The antidisestablishmentarianism movement gained considerable momentum during the parliamentary proceedings.",
        "E2a": "The orchestra's conduct during the ceremony was praised by the audience.",
        "E2b": "The researchers will conduct an experiment to evaluate the proposed method.",
        "E3": "The rhyth m of the orchestra changed unexpectedly.",
        "E4": "The meeting will start at 10:45 AM on 12 November 2026, so andaroo should arrive on time."
    },
    "Telugu": {
        "T1": "న్యూరోఫార్మకోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
        "T2a": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
        "T2b": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
        "T3": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
        "T4": "ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉండండి."
    }
}

for lang, sentences in TEST_SENTENCES.items():
    ckpt = torch.load(
        ROOT / "checkpoints_v2" / lang / "best.pt",
        map_location="cpu",
        weights_only=False
    )

    vocab = ckpt["vocab"]

    print(f"\n{'=' * 65}")
    print(lang.upper())
    print(f"{'=' * 65}")

    for sentence_id, original in sentences.items():
        processed = normalize_text(original, lang)
        missing = sorted(set(processed) - set(vocab))

        print(f"\n{sentence_id}")
        print("Original:  ", original)
        print("Processed: ", processed)
        print("Missing:   ", missing if missing else "None")

In [ ]:
import csv
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/kaggle/working/fastspeech2_project")
OUT = ROOT / "stage04_v2"
WAV_DIR = OUT / "wavs"
PLOT_DIR = OUT / "variance_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "English": english_v2,
    "Telugu": telugu_v2
}

rows = []

for lang, sentences in TEST_SENTENCES.items():
    net = MODELS[lang]
    net.eval()
    device = next(net.parameters()).device

    ckpt = torch.load(
        ROOT / "checkpoints_v2" / lang / "best.pt",
        map_location="cpu",
        weights_only=False
    )
    vocab = ckpt["vocab"]

    for sid, original in sentences.items():
        processed = normalize_text(original, lang)
        token_ids = [vocab[c] for c in processed]
        tokens = torch.tensor(
            [token_ids], dtype=torch.long, device=device
        )

        with torch.inference_mode():
            output = net(tokens)

        n = len(token_ids)

        # Extract the actual V2 predictor outputs
        logdur = output["logdur"][0].detach().cpu().numpy()[:n]
        pitch = output["pitch"][0].detach().cpu().numpy()[:n]
        energy = output["energy"][0].detach().cpu().numpy()[:n]

        # Convert predicted log-duration into approximate frame counts.
        # Assumes the model uses log(1 + duration) targets.
        durations = np.maximum(np.expm1(logdur), 0)

        x = np.arange(n)

        fig, axes = plt.subplots(
            3, 1,
            figsize=(16, 11),
            gridspec_kw={"hspace": 0.6}
        )

        axes[0].bar(x, durations, color="#4472C4", width=0.8)
        axes[0].set_title("Predicted Duration", fontsize=13, pad=14)
        axes[0].set_ylabel("Approx. mel frames")

        axes[1].plot(x, pitch, color="#E67E22", linewidth=1.8)
        axes[1].set_title("Predicted Pitch", fontsize=13, pad=14)
        axes[1].set_ylabel("Normalized pitch")

        axes[2].plot(x, energy, color="#27AE60", linewidth=1.8)
        axes[2].set_title("Predicted Energy", fontsize=13, pad=14)
        axes[2].set_ylabel("Normalized energy")
        axes[2].set_xlabel("Input character index")

        for ax in axes:
            ax.grid(alpha=0.2)
            ax.set_xlim(-0.5, n - 0.5)

        fig.suptitle(
            f"{lang} — {sid} | FastSpeech 2 V2 Variance Predictions",
            fontsize=17,
            fontweight="bold",
            y=0.975
        )

        fig.subplots_adjust(
            left=0.09,
            right=0.96,
            top=0.91,
            bottom=0.08
        )

        plot_path = PLOT_DIR / f"{sid}_variance.png"
        fig.savefig(
            plot_path,
            dpi=250,
            bbox_inches="tight",
            facecolor="white"
        )
        plt.close(fig)

        wav_path = WAV_DIR / f"{sid}_{lang.lower()}.wav"

        rows.append({
            "id": sid,
            "language": lang,
            "original_text": original,
            "processed_text": processed,
            "input_characters": n,
            "generated_frames": int(output["mel_lengths"][0]),
            "wav_path": str(wav_path),
            "variance_plot": str(plot_path)
        })

        print(f"{sid}: saved 3 variance plots")

csv_path = OUT / "synthesis_metadata.csv"

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("\nSTAGE 04 COMPLETE")
print("WAV files:", len(list(WAV_DIR.glob("*.wav"))))
print("Variance figures:", len(list(PLOT_DIR.glob("*.png"))))
print("Individual variance plots:", len(rows) * 3)
print("Metadata:", csv_path)

In [ ]:
import json
import csv
import warnings
import numpy as np
import torch
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Audio

ROOT = Path("/kaggle/working/fastspeech2_project")
OUT = ROOT / "stage04_v2"
WAV_DIR = OUT / "wavs"
PLOT_DIR = OUT / "variance_plots"

WAV_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "English": english_v2,
    "Telugu": telugu_v2
}

def get_prediction(output, names):
    for name in names:
        if name in output:
            value = output[name]
            if isinstance(value, torch.Tensor):
                return value.detach().float().cpu().numpy()
    return None

def plot_variance(lang, sid, text, durations, pitch, energy):
    fig, axes = plt.subplots(
        3, 1, figsize=(15, 10),
        gridspec_kw={"hspace": 0.55}
    )

    x = np.arange(len(durations))

    axes[0].bar(x, durations, color="#4472C4")
    axes[0].set_title("Predicted Phoneme/Character Durations", pad=12)
    axes[0].set_ylabel("Mel frames")

    axes[1].plot(x, pitch, color="#E67E22", linewidth=1.8)
    axes[1].set_title("Predicted Pitch", pad=12)
    axes[1].set_ylabel("Pitch predictor output")

    axes[2].plot(x, energy, color="#27AE60", linewidth=1.8)
    axes[2].set_title("Predicted Energy", pad=12)
    axes[2].set_ylabel("Energy predictor output")
    axes[2].set_xlabel("Input character index")

    for ax in axes:
        ax.grid(alpha=0.2)

    fig.suptitle(
        f"{lang} | {sid} | FastSpeech 2 V2 Variance Predictions",
        fontsize=16, fontweight="bold", y=0.98
    )

    fig.subplots_adjust(
        left=0.09, right=0.96,
        top=0.90, bottom=0.08
    )

    path = PLOT_DIR / f"{sid}_variance.png"
    fig.savefig(path, dpi=250, bbox_inches="tight")
    plt.close(fig)
    return path

rows = []

for lang, sentences in TEST_SENTENCES.items():
    print(f"\n{'='*60}\n{lang}\n{'='*60}")

    net = MODELS[lang]
    net.eval()
    device = next(net.parameters()).device

    checkpoint = torch.load(
        ROOT / "checkpoints_v2" / lang / "best.pt",
        map_location="cpu",
        weights_only=False
    )

    vocab = checkpoint["vocab"]
    stats = checkpoint["stats"]

    for sid, original in sentences.items():
        processed = normalize_text(original, lang)
        missing = sorted(set(processed) - set(vocab))

        if missing:
            raise ValueError(f"{sid}: Unknown characters {missing}")

        token_ids = [vocab[c] for c in processed]
        tokens = torch.tensor(
            token_ids, dtype=torch.long, device=device
        ).unsqueeze(0)

        with torch.inference_mode():
            output = net(tokens)

        mel = output["post_mel"][0].detach().float().cpu().numpy()

        # Trim to the model's predicted output length
        length = int(output["mel_lengths"][0])
        mel = mel[:length]

        wav = v2_mel_to_wav(mel, stats, n_iter=48)

        wav_path = WAV_DIR / f"{sid}_{lang.lower()}.wav"
        sf.write(wav_path, wav, 22050)

        # Find variance predictions using common naming conventions
        durations = get_prediction(
            output,
            ["duration_pred", "pred_duration", "durations",
             "duration", "duration_prediction", "dur_pred"]
        )
        pitch = get_prediction(
            output,
            ["pitch_pred", "pred_pitch", "pitch",
             "pitch_prediction"]
        )
        energy = get_prediction(
            output,
            ["energy_pred", "pred_energy", "energy",
             "energy_prediction"]
        )

        if durations is None or pitch is None or energy is None:
            print(f"\n{sid}: Audio saved, but variance keys need checking.")
            print("Available output keys:", list(output.keys()))
            print("WAV:", wav_path)
            continue

        durations = np.asarray(durations).reshape(-1)[:len(token_ids)]
        pitch = np.asarray(pitch).reshape(-1)[:len(token_ids)]
        energy = np.asarray(energy).reshape(-1)[:len(token_ids)]

        plot_path = plot_variance(
            lang, sid, processed, durations, pitch, energy
        )

        rows.append({
            "id": sid,
            "language": lang,
            "original_text": original,
            "processed_text": processed,
            "input_characters": len(token_ids),
            "generated_frames": length,
            "wav_path": str(wav_path),
            "variance_plot": str(plot_path)
        })

        print(
            f"{sid}: {len(token_ids)} chars, "
            f"{length} frames, {len(wav)/22050:.2f}s"
        )

csv_path = OUT / "synthesis_metadata.csv"

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "id", "language", "original_text", "processed_text",
            "input_characters", "generated_frames",
            "wav_path", "variance_plot"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

print("\n" + "="*60)
print("STAGE 04 SUMMARY")
print("="*60)
print("WAV files:", len(list(WAV_DIR.glob("*.wav"))))
print("Variance plots:", len(list(PLOT_DIR.glob("*.png"))))
print("Metadata:", csv_path)
print("Output folder:", OUT)

In [ ]:
from pathlib import Path
from IPython.display import display, Audio, Markdown
import soundfile as sf

WAV_DIR = Path(
    "/kaggle/working/fastspeech2_project/stage04_v2/wavs"
)

for lang, ids in {
    "English": ["E1", "E2a", "E2b", "E3", "E4"],
    "Telugu": ["T1", "T2a", "T2b", "T3", "T4"]
}.items():

    display(Markdown(f"## {lang} — Generated Speech"))

    for sid in ids:
        path = WAV_DIR / f"{sid}_{lang.lower()}.wav"

        if path.exists():
            audio, sr = sf.read(path)
            duration = len(audio) / sr

            display(Markdown(
                f"**{sid}** — {duration:.2f} seconds"
            ))

            display(Audio(
                filename=str(path),
                rate=sr
            ))
        else:
            print("Missing:", path)

In [ ]:
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Audio

# One training sample
ds = AcousticDatasetV2("English", "train")
idx = next(
    i for i in range(len(ds))
    if 120 <= len(ds[i][4]) <= 250
)

tok, dur, pit, eng, mel, voi = ds[idx]

# Create a fresh model
net = FastSpeech2V2(
    vocab_size=len(ds.vocab) + 1
).to(DEVICE)

net.train()

opt = torch.optim.AdamW(
    net.parameters(),
    lr=5e-4,
    weight_decay=0
)

tokens = tok[None].to(DEVICE)
durations = dur[None].to(DEVICE)
pitch = pit[None].to(DEVICE)
energy = eng[None].to(DEVICE)
voicing = voi[None].to(DEVICE)
target = mel[None].to(DEVICE)

losses = []

# Train on the SAME sentence repeatedly
for step in range(1, 1001):
    net.train()
    opt.zero_grad()

    out = net(
        tokens,
        durations,
        pitch,
        energy,
        voicing
    )

    loss = (
        (out["mel"] - target).abs().mean()
        + (out["post_mel"] - target).abs().mean()
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    opt.step()

    losses.append(float(loss.detach()))

    if step % 100 == 0:
        print(
            f"Step {step:4d}/1000 | "
            f"Mel reconstruction loss: {losses[-1]:.4f}"
        )

# Save separately from existing checkpoints
from pathlib import Path

debug_dir = Path(
    "/kaggle/working/fastspeech2_project/debug_overfit"
)
debug_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    net.state_dict(),
    debug_dir / "english_one_sentence.pt"
)

# Evaluate
net.eval()

with torch.inference_mode():
    out = net(
        tokens,
        durations,
        pitch,
        energy,
        voicing
    )

pred = out["post_mel"][0].detach().cpu().numpy()

# Listen
print("\nORIGINAL MEL RECONSTRUCTION")
display(Audio(
    v2_mel_to_wav(mel.numpy(), STATS["English"], n_iter=64),
    rate=22050
))

print("\nONE-SENTENCE OVERFITTED MODEL")
display(Audio(
    v2_mel_to_wav(pred, STATS["English"], n_iter=64),
    rate=22050
))

# Plot learning curve
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.title("English — One-Sentence Overfitting Test")
plt.xlabel("Optimization step")
plt.ylabel("Mel reconstruction loss")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    debug_dir / "one_sentence_loss.png",
    dpi=200
)
plt.show()

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Subset
from IPython.display import Audio, display
from pathlib import Path
import matplotlib.pyplot as plt

ds = AcousticDatasetV2("English", "train")

# Choose 16 reasonably short samples
indices = [
    i for i in range(len(ds))
    if 100 <= len(ds[i][4]) <= 250
][:16]

assert len(indices) == 16

small_ds = Subset(ds, indices)

loader = DataLoader(
    small_ds,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_v2
)

net16 = FastSpeech2V2(
    vocab_size=len(ds.vocab) + 1
).to(DEVICE)

opt = torch.optim.AdamW(
    net16.parameters(),
    lr=3e-4,
    weight_decay=0
)

losses = []

# 100 epochs × 4 batches = 400 optimization steps
for epoch in range(1, 101):
    net16.train()
    epoch_losses = []

    for batch in loader:
        b = tuple(x.to(DEVICE) for x in batch)

        opt.zero_grad(set_to_none=True)

        out = net16(
            b[0], b[1], b[2], b[3], b[5]
        )

        # Use the same full V2 objective
        loss, parts = loss_v2(out, b)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            net16.parameters(), 1.0
        )
        opt.step()

        epoch_losses.append(float(loss.detach()))

    avg_loss = float(np.mean(epoch_losses))
    losses.append(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/100 | Loss: {avg_loss:.4f}")

# Save separately
debug_dir = Path(
    "/kaggle/working/fastspeech2_project/debug_overfit"
)
debug_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    net16.state_dict(),
    debug_dir / "english_16_sentences.pt"
)

# Listen to two training sentences
net16.eval()

for sample_idx in [0, 1]:
    tok, dur, pit, eng, mel, voi = small_ds[sample_idx]

    with torch.inference_mode():
        out = net16(
            tok[None].to(DEVICE),
            dur[None].to(DEVICE),
            pit[None].to(DEVICE),
            eng[None].to(DEVICE),
            voi[None].to(DEVICE)
        )

    pred = out["post_mel"][0].cpu().numpy()

    print(f"\nTRAINING SENTENCE {sample_idx + 1}")
    print("Original:")
    display(Audio(
        v2_mel_to_wav(
            mel.numpy(), STATS["English"], n_iter=48
        ),
        rate=22050
    ))

    print("Model prediction:")
    display(Audio(
        v2_mel_to_wav(
            pred, STATS["English"], n_iter=48
        ),
        rate=22050
    ))

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.title("English — 16-Sentence Overfitting Experiment")
plt.xlabel("Epoch")
plt.ylabel("Combined V2 Loss")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    debug_dir / "english_16_sentences_loss.png",
    dpi=200
)
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path

path = Path(
    "/kaggle/working/fastspeech2_project/English/train_manifest.csv"
)

df = pd.read_csv(path)

print("Manifest columns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

speaker_cols = [
    c for c in df.columns
    if "speaker" in c.lower() or "spk" in c.lower()
]

if speaker_cols:
    for col in speaker_cols:
        print(f"\n{col}: {df[col].nunique()} unique speakers")
        print(df[col].value_counts().head(15))
else:
    print("\nNo explicit speaker column found.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/kaggle/working/fastspeech2_project")
results = []

for lang in ["English", "Telugu"]:
    ds = AcousticDatasetV2(lang, "train")

    for i in range(len(ds)):
        tok, dur, pit, eng, mel, voi = ds[i]

        d = dur.numpy()
        n = len(d)

        results.append({
            "language": lang,
            "index": i,
            "characters": n,
            "mel_frames": len(mel),
            "zero_durations": int((d == 0).sum()),
            "zero_percent": round(100 * (d == 0).mean(), 2),
            "one_frame_percent": round(100 * (d <= 1).mean(), 2),
            "max_duration": int(d.max()),
            "mean_duration": round(float(d.mean()), 2),
            "duration_sum_valid": int(d.sum()) == len(mel)
        })

df = pd.DataFrame(results)

for lang in ["English", "Telugu"]:
    x = df[df.language == lang]

    print(f"\n{'='*50}")
    print(lang.upper())
    print(f"{'='*50}")

    print("Samples:", len(x))
    print("Mean zero-duration %:", round(x.zero_percent.mean(), 2))
    print("Mean <=1-frame %:", round(x.one_frame_percent.mean(), 2))
    print("Samples with >20% zero durations:",
          int((x.zero_percent > 20).sum()))
    print("Samples with duration >45 frames:",
          int((x.max_duration > 45).sum()))
    print("Invalid duration sums:",
          int((~x.duration_sum_valid).sum()))

    print("\nWorst 5 samples:")
    display(
        x.sort_values("zero_percent", ascending=False)
         .head(5)
    )

OUT = ROOT / "training_visualizations"
OUT.mkdir(exist_ok=True)

df.to_csv(OUT / "alignment_quality_analysis.csv", index=False)

print("\nSaved alignment analysis.")

In [ ]:
import numpy as np
import pandas as pd

rows = []

for lang in ["English", "Telugu"]:
    ds = AcousticDatasetV2(lang, "train")

    for i in range(len(ds)):
        tok, dur, pit, eng, mel, voi = ds[i]
        d = dur.numpy()

        rows.append({
            "language": lang,
            "sample": i,
            "max_duration": int(d.max()),
            "over_45": int((d > 45).sum()),
            "over_80": int((d > 80).sum()),
            "over_120": int((d > 120).sum()),
            "frames_lost_if_capped": int(
                np.maximum(d - 45, 0).sum()
            ),
            "total_frames": len(mel)
        })

audit = pd.DataFrame(rows)

for lang in ["English", "Telugu"]:
    x = audit[audit.language == lang].copy()

    print(f"\n=== {lang.upper()} ===")
    print("Characters >45 frames:", x.over_45.sum())
    print("Characters >80 frames:", x.over_80.sum())
    print("Characters >120 frames:", x.over_120.sum())

    lost = x.frames_lost_if_capped.sum()
    total = x.total_frames.sum()

    print("Frames exceeding 45:", lost)
    print("Percentage of all frames:", round(100 * lost / total, 2), "%")

    print("\nLongest-duration samples:")
    display(
        x.sort_values("max_duration", ascending=False).head(5)
    )

In [ ]:
from pathlib import Path
import numpy as np

ds = AcousticDatasetV2("English", "train")

thresholds = [80, 120, 200, 300]

for limit in thresholds:
    keep = []
    excluded = []

    for i in range(len(ds)):
        durations = ds[i][1].numpy()

        if durations.max() <= limit:
            keep.append(i)
        else:
            excluded.append(i)

    print(
        f"Maximum duration <= {limit:3d} frames | "
        f"Keep: {len(keep):4d} | "
        f"Exclude: {len(excluded):4d} | "
        f"Retained: {100 * len(keep) / len(ds):.1f}%"
    )

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

ds_v3 = AcousticDatasetV2("English", "train")

keep_indices = []

for i in range(len(ds_v3)):
    durations = ds_v3[i][1].numpy()

    if durations.max() <= 120:
        keep_indices.append(i)

train_v3 = Subset(ds_v3, keep_indices)

loader_v3 = DataLoader(
    train_v3,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_v2,
    num_workers=2,
    pin_memory=True
)

print("V3 training samples:", len(train_v3))
print("Batches per epoch:", len(loader_v3))

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from pathlib import Path

ROOT = Path("/kaggle/working/fastspeech2_project")
V4_DIR = ROOT / "experiments" / "english_v4"
V4_DIR.mkdir(parents=True, exist_ok=True)

ds128 = AcousticDatasetV2("English", "train")

# Reproducible sample selection
rng = np.random.default_rng(42)

eligible = [
    i for i in range(len(ds128))
    if 100 <= len(ds128[i][4]) <= 350
    and int(ds128[i][1].max()) <= 120
]

assert len(eligible) >= 128, "Not enough eligible samples"

indices128 = rng.choice(
    eligible, size=128, replace=False
).tolist()

small128 = Subset(ds128, indices128)

loader128 = DataLoader(
    small128,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_v2,
    num_workers=2,
    pin_memory=True
)

np.save(V4_DIR / "selected_indices.npy", indices128)

print("Selected samples:", len(small128))
print("Batches per epoch:", len(loader128))
print("Output directory:", V4_DIR)

In [ ]:
from IPython.display import Audio, display

net128.eval()

for j in [0, 1]:
    tok, dur, pit, eng, mel, voi = small128[j]

    t = tok[None].to(DEVICE)

    with torch.inference_mode():
        teacher = net128(
            t,
            dur[None].to(DEVICE),
            pit[None].to(DEVICE),
            eng[None].to(DEVICE),
            voi[None].to(DEVICE)
        )
        free = net128(t)

    print(f"\n=== TRAINING SAMPLE {j + 1} ===")

    print("1. Original mel reconstruction")
    display(Audio(
        v2_mel_to_wav(
            mel.numpy(), STATS["English"], n_iter=48
        ),
        rate=22050
    ))

    print("2. Teacher-forced reconstruction")
    display(Audio(
        v2_mel_to_wav(
            teacher["post_mel"][0].cpu().numpy(),
            STATS["English"], n_iter=48
        ),
        rate=22050
    ))

    print("3. Fully predicted synthesis")
    display(Audio(
        v2_mel_to_wav(
            free["post_mel"][0].cpu().numpy(),
            STATS["English"], n_iter=48
        ),
        rate=22050
    ))

In [ ]:
from pathlib import Path
import zipfile

root = Path("/kaggle/working/fastspeech2_project")
zip_path = Path("/kaggle/working/fastspeech2_v2_submission_backup.zip")

folders = [
    "checkpoints_v2",
    "stage04_v2",
    "training_visualizations"
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        p = root / folder
        if p.exists():
            for file in p.rglob("*"):
                if file.is_file():
                    z.write(file, file.relative_to(root))

    for lang in ["English", "Telugu"]:
        for name in ["vocab.json", "feature_config.json"]:
            file = root / "features" / lang / name
            if file.exists():
                z.write(file, file.relative_to(root))

print("Backup created:", zip_path)
print("Size:", round(zip_path.stat().st_size / 1024**2, 2), "MB")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

net128.eval()

tok, dur, pit, eng, mel, voi = small128[0]

with torch.inference_mode():
    out = net128(
        tok[None].to(DEVICE),
        dur[None].to(DEVICE),
        pit[None].to(DEVICE),
        eng[None].to(DEVICE),
        voi[None].to(DEVICE)
    )

target = mel.numpy()
pred = out["post_mel"][0].cpu().numpy()

print("Target mel shape:", target.shape)
print("Predicted mel shape:", pred.shape)
print("Mel MAE:", np.mean(np.abs(target - pred)))
print("Target standard deviation:", target.std())
print("Prediction standard deviation:", pred.std())

fig, ax = plt.subplots(2, 1, figsize=(13, 6))

vmin, vmax = np.percentile(target, [2, 98])

ax[0].imshow(
    target.T, origin="lower", aspect="auto",
    vmin=vmin, vmax=vmax
)
ax[0].set_title("Actual mel spectrogram")

ax[1].imshow(
    pred.T, origin="lower", aspect="auto",
    vmin=vmin, vmax=vmax
)
ax[1].set_title("V4 predicted mel spectrogram")

plt.tight_layout()
plt.show()